# Projeto (Seção 7) — enunciado

Foi aqui que paramos no projeto da seção anterior:

In [1]:
from datetime import date
from enum import Enum
from uuid import UUID, uuid4
from pydantic import BaseModel, ConfigDict, Field, UUID4, field_serializer
from pydantic.alias_generators import to_camel


class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"


class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=to_camel,
    )

    id_: UUID4 = Field(alias="id", default_factory=uuid4)
    manufacturer: str
    series_name: str
    type_: AutomobileType = Field(alias="type")
    is_electric: bool = False
    manufactured_date: date = Field(validation_alias="completionDate", ge=date(1980, 1, 1))
    base_msrp_usd: float = Field(
        validation_alias="msrpUSD",
        serialization_alias="baseMSRPUSD"
    )
    vin: str
    number_of_doors: int = Field(
        default=4,
        validation_alias="doors",
        ge=2,
        le=4,
        multiple_of=2,
    )
    registration_country: str | None = None
    license_plate: str | None = None

    @field_serializer("manufactured_date", when_used="json-unless-none")
    def serialize_date(self, value: date) -> str:
        return value.strftime("%Y/%m/%d")

Crie um tipo anotado chamado `BoundedString` que define uma string com no mínimo 2 e no máximo 50 caracteres.

Modifique o `Automobile` para usar esse tipo anotado nos campos:
- `manufacturer`
- `series_name`
- `vin`
- `registration_country`
- `license_plate`

Crie um tipo anotado chamado `BoundedList` que usa uma variável de tipo para definir uma lista com no mínimo `1` e no máximo `5` elementos.

Com esse tipo anotado, adicione um campo novo ao modelo:
- nome do campo: `top_features`
- posicionado logo antes do campo `vin`
- desserializado de e serializado para `topFeatures`
- deve ser uma lista limitada de strings, que por sua vez devem ter no mínimo `2` e no máximo `50` caracteres (dica: use o `BoundedString`
  como tipo ao definir o campo com `BoundedList`)
- opcional, com default `None`

Use estes dados para testar o modelo:

In [2]:
data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4 Competition xDrive",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "topFeatures": ["6 cylinders", "all-wheel drive", "convertible"],
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "France",
    "licensePlate": "AAA-BBB"
}

expected_serialized_by_alias = {
    'id': UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'),
    'manufacturer': 'BMW',
    'seriesName': 'M4 Competition xDrive',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'topFeatures': ['6 cylinders', 'all-wheel drive', 'convertible'],
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': 'France',
    'licensePlate': 'AAA-BBB'
}

Teste também que você recebe erros de validação ao desserializar dados que não respeitam as restrições.

> Sugestão: crie modelos simples, de um campo só, para testar os tipos anotados e confirmar que funcionam. Depois, use-os no modelo principal.

➡️ Soluções: [`06_projeto_solucao_instrutor`](06_projeto_solucao_instrutor.ipynb) · [`07_projeto_solucao_comentada`](07_projeto_solucao_comentada.ipynb)